In [3]:
import json
from pathlib import Path

import numpy as np
import polars as pl

split = Path("data/competition/competition/train")
feed = (
    pl.scan_parquet(str(split / "impressions/part-*.parquet"))
    .head(1)
    .collect()
    .row(0, named=True)
)

embedding_dir = split / "item_embeddings"
meta = json.loads((embedding_dir / "data.meta").read_text())
embeddings = np.memmap(
    embedding_dir / "data.memmap",
    dtype=meta["dtype"],
    mode="r",
    shape=tuple(meta["shape"]),
    order=meta["order"],
)

indices = np.asarray(feed["item_id__embedding_idx"], dtype=np.int64)
present = indices >= 0
candidate_vectors = np.zeros((len(indices), 128), dtype=np.float32)
candidate_vectors[present] = embeddings[indices[present]].astype(np.float32)
user_vector = np.asarray(feed["user_embedding"], dtype=np.float32)